In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier,VotingClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import GridSearchCV,RandomizedSearchCV
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
df=pd.read_csv('adult.csv')
x=df.drop('income',axis=1)
y=df['income']
df.replace(' ?', pd.NA, inplace=True)
y = y.apply(lambda x: 1 if '>50K' in x else 0)
num_columns=x.select_dtypes(include=['int64','float64']).columns
cat_columns=x.select_dtypes(include=['object']).columns
num_pipline=Pipeline([
    ('impute',SimpleImputer(strategy='mean')),
    ('Scaler',StandardScaler())
])
cat_pipline=Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('Encoder',OneHotEncoder(handle_unknown='ignore'))
])
preproessor=ColumnTransformer([
    ('num',num_pipline,num_columns),
    ('cat',cat_pipline,cat_columns)
])
lg=LogisticRegression(max_iter=200)
rf=RandomForestClassifier(n_estimators=100,max_depth=5,random_state=42)
dt=DecisionTreeClassifier(max_depth=5)

lg_pipline=Pipeline([('preprocessing',preproessor),('model',lg)])
rf_pipline=Pipeline([('preprocessing',preproessor),('model',rf)])
dt_pipline=Pipeline([('preprocessing',preproessor),('model',dt)])
voting=VotingClassifier(
    estimators=[
        ('lg',lg_pipline),
         ('rf',rf_pipline),
          ('dt',dt_pipline),
    ]
)
X_train,X_test,y_train ,y_test=train_test_split(x,y,random_state=42,test_size=0.2)

param_grid={
'rf__model__n_estimators':[100,200],
'rf__model__max_depth':[5,10],
'dt__model__max_depth':[5,10],
'lg__model__C':[0.1,1,10]
}
grid=GridSearchCV(
    voting,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid.fit(X_train,y_train)

print("Best Parameter",grid.best_params_)
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)


print("Accuracy:",accuracy_score(y_test,y_pred))
cm=confusion_matrix(y_pred,y_test)
print("Confusion Matrix:",cm)
print("Classification Report:",classification_report( y_test,y_pred))

sns.heatmap(cm,annot=True,fmt='d')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()
joblib.dump(best_model,"income_model.pkl")

C:\Users\GMB\AppData\Local\Temp\ipykernel_7768\1792646580.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_columns=x.select_dtypes(include=['object']).columns
